# 06 -- Experiment Tracking and Comparison

When tuning a retrieval pipeline you run dozens of experiments: different retrievers, chunk sizes, rerankers, fusion weights. This notebook shows how to save, load, and compare experiments using RankFlow's built-in lightweight experiment registry -- no MLflow or W&B required.

**Key classes:** `Experiment`, `ExperimentStore`, `compare_experiments`, `ComparisonReport`.

In [ ]:
%matplotlib inline

import tempfile
from pathlib import Path

import numpy as np

from rankflow import (
    RankFlow,
    Experiment,
    ExperimentStore,
    compare_experiments,
)

## Pipeline config metadata

Every `RankFlow` can carry a `pipeline_config` dict describing the settings that produced it. This metadata travels through JSON export/import and is used for automatic config diffing when comparing experiments.

In [ ]:
rf = RankFlow(
    ranks=np.array([[0, 1, 2], [1, 0, 2]]),
    step_labels=["BM25", "Reranker"],
    chunk_labels=["doc_a", "doc_b", "doc_c"],
    relevant_chunks=["doc_a"],
    pipeline_config={"retriever": "bm25", "top_k": 100, "chunk_size": 512},
)
print(f"Config: {rf.pipeline_config}")

## Creating synthetic experiment data

We'll simulate two experiments: a BM25 baseline and a cross-encoder challenger, each evaluated on 20 queries.

In [ ]:
rng = np.random.default_rng(42)
n_queries = 20
n_docs = 15
chunk_labels = [f"doc_{i}" for i in range(n_docs)]

def make_rankflows(seed, improvement_factor=0):
    """Generate rankflows with optional improvement at the reranking step."""
    rng_local = np.random.default_rng(seed)
    rfs = []
    for q in range(n_queries):
        base = rng_local.permutation(n_docs)
        reranked = np.argsort(np.argsort(
            base + rng_local.normal(0, 3 - improvement_factor, n_docs)
        ))
        ranks = np.array([base, reranked])
        rel = rng_local.choice(chunk_labels, size=2, replace=False).tolist()
        rf = RankFlow(
            ranks=ranks,
            step_labels=["BM25", "Reranker"],
            chunk_labels=chunk_labels,
            relevant_chunks=rel,
        )
        rf.query_label = f"query_{q}"
        rfs.append(rf)
    return rfs

baseline_rfs = make_rankflows(seed=42, improvement_factor=0)
challenger_rfs = make_rankflows(seed=42, improvement_factor=1.5)

print(f"Baseline: {len(baseline_rfs)} queries")
print(f"Challenger: {len(challenger_rfs)} queries")

## Saving experiments

An `ExperimentStore` is just a directory of JSON files. Each `Experiment` has a name, config dict, tags, and a list of `RankFlow` objects.

In [ ]:
store_dir = Path(tempfile.mkdtemp()) / "experiments"
store = ExperimentStore(store_dir)

baseline_exp = Experiment(
    name="bm25-baseline",
    config={"retriever": "bm25", "top_k": 100, "reranker": "none"},
    rankflows=baseline_rfs,
    tags=["baseline", "v1"],
    description="BM25 with no reranking",
)
store.save(baseline_exp)

challenger_exp = Experiment(
    name="cross-encoder-v1",
    config={"retriever": "bm25", "top_k": 100, "reranker": "cross-encoder", "model": "ms-marco-MiniLM"},
    rankflows=challenger_rfs,
    tags=["challenger", "v1"],
    description="BM25 + cross-encoder reranker",
)
store.save(challenger_exp)

print(f"Saved to: {store_dir}")

## Listing experiments

In [ ]:
for exp_info in store.list():
    config_str = ", ".join(f"{k}={v}" for k, v in list(exp_info['config'].items())[:3])
    print(f"  {exp_info['name']:25s} | {exp_info['n_queries']} queries | {config_str}")

## Loading and inspecting an experiment

In [ ]:
loaded = store.load("bm25-baseline")
print(f"Name: {loaded.name}")
print(f"Queries: {loaded.n_queries}")
print(f"Tags: {loaded.tags}")
print(f"Config: {loaded.config}")

# Quick headline metrics
summary = loaded.metrics_summary(k=5)
for metric, value in summary.items():
    print(f"  {metric}: {value:.3f}")

## Comparing two experiments

`compare_experiments()` matches queries by label, computes per-metric deltas with statistical significance (paired t-test), and counts wins/losses/ties.

In [ ]:
report = compare_experiments(baseline_exp, challenger_exp, k=5)

print(f"Baseline:   {report.baseline_name}")
print(f"Challenger: {report.challenger_name}")
print(f"Matched queries: {len(report.per_query)}")
print(f"\nWin/Loss/Tie: {report.wins}W / {report.losses}L / {report.ties}T")
print(f"Win rate: {report.win_rate:.0%}")

### Configuration diff

Automatically identifies what changed between two experiment configs.

In [ ]:
print("Config differences:")
for key, vals in report.config_diff.items():
    print(f"  {key}: {vals['baseline']} -> {vals['challenger']}")

### Metric deltas with statistical significance

In [ ]:
print(f"{'Metric':<20s} {'Baseline':>10s} {'Challenger':>10s} {'Delta':>8s} {'p-value':>8s} {'Sig?':>5s}")
print("-" * 65)
for metric, data in report.metric_deltas.items():
    sig = "*" if data['p_value'] < 0.05 else ""
    print(f"{metric:<20s} {data['baseline_mean']:>10.3f} {data['challenger_mean']:>10.3f} "
          f"{data['delta']:>+8.3f} {data['p_value']:>8.3f} {sig:>5s}")

### Identifying regressions and improvements

In [ ]:
regressions = report.regression_queries("ndcg_at_k")
improved = report.improved_queries("ndcg_at_k")

print(f"Improved: {len(improved)} queries")
print(f"Regressed: {len(regressions)} queries")

if regressions:
    print("\nWorst regressions:")
    sorted_reg = sorted(regressions, key=lambda q: q['ndcg_at_k_delta'])
    for q in sorted_reg[:3]:
        print(f"  {q['query_label']}: {q['ndcg_at_k_baseline']:.3f} -> "
              f"{q['ndcg_at_k_challenger']:.3f} (delta={q['ndcg_at_k_delta']:+.3f})")

### Drill into a specific query

In [ ]:
# Pick the first query and compare visually
q_label = baseline_exp.rankflows[0].query_label
print(f"Comparing query: {q_label}")

RankFlow.compare(
    baseline_exp.rankflows[0],
    challenger_exp.rankflows[0],
    labels=(baseline_exp.name, challenger_exp.name),
)

## Web UI

For interactive exploration, RankFlow includes a Streamlit-based web UI. Launch it from the command line:

```bash
pip install rankflow[ui]
rankflow ui ./experiments
```

The UI provides four views:

1. **Experiment List** -- browse all saved experiments, filter by tag, preview metrics
2. **Comparison Dashboard** -- config diff, metric deltas with significance, win/loss/tie
3. **Query Explorer** -- per-query metrics table with drill-down to individual rank plots
4. **Deep Dive** -- full BatchRankFlow dashboard for a single experiment

---

**Previous:** [05 -- Adapters and Export](05_adapters_and_export.ipynb)

**Next:** [07 -- Web UI Walkthrough](07_web_ui_walkthrough.ipynb) -- generate synthetic data and explore the interactive Streamlit dashboard.

**Start from the beginning:** [01 -- Quick Start](01_quickstart.ipynb)